# Train Pre-Match Feature Parsing

Parse all YAML files from the train split and build a pre-match dataset with historical team features.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd

train_dir = Path("../data/splits/pre_match_eval/train")
yaml_files = sorted(train_dir.glob("*.yaml"))

len(yaml_files)

In [2]:
def parse_match_metadata(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    info = data["info"]
    teams = info["teams"]
    outcome = info.get("outcome", {})
    toss = info.get("toss", {})

    return {
        "match_id": file_path.stem,
        "match_date": info["dates"][0],
        "season": int(str(info["dates"][0])[:4]),
        "team1": teams[0],
        "team2": teams[1],
        "venue": info.get("venue"),
        "toss_winner": toss.get("winner"),
        "toss_decision": toss.get("decision"),
        "winner": outcome.get("winner")
    }

matches = [parse_match_metadata(path) for path in yaml_files]
matches_df = pd.DataFrame(matches)
matches_df["match_date"] = pd.to_datetime(matches_df["match_date"])
matches_df = matches_df.sort_values(["match_date", "match_id"]).reset_index(drop=True)

matches_df.head()

In [3]:
def win_pct_last_n(history, team, n=5):
    team_history = history[(history["team1"] == team) | (history["team2"] == team)].tail(n)
    if team_history.empty:
        return 0.5
    return (team_history["winner"] == team).mean()

def head_to_head_win_pct(history, team, opponent):
    h2h = history[
        ((history["team1"] == team) & (history["team2"] == opponent)) |
        ((history["team1"] == opponent) & (history["team2"] == team))
    ]
    if h2h.empty:
        return 0.5
    return (h2h["winner"] == team).mean()

def venue_win_pct(history, team, venue):
    venue_history = history[
        (((history["team1"] == team) | (history["team2"] == team)) & (history["venue"] == venue))
    ]
    if venue_history.empty:
        return 0.5
    return (venue_history["winner"] == team).mean()

feature_rows = []

for idx, row in matches_df.iterrows():
    history = matches_df.iloc[:idx]
    team1 = row["team1"]
    team2 = row["team2"]
    venue = row["venue"]

    feature_rows.append({
        "match_id": row["match_id"],
        "match_date": row["match_date"],
        "team1": team1,
        "team2": team2,
        "venue": venue,
        "season": row["season"],
        "toss_winner": row["toss_winner"],
        "toss_decision": row["toss_decision"],
        "team1_win_pct_last_5": win_pct_last_n(history, team1),
        "team2_win_pct_last_5": win_pct_last_n(history, team2),
        "team1_head_to_head_win_pct": head_to_head_win_pct(history, team1, team2),
        "team2_head_to_head_win_pct": head_to_head_win_pct(history, team2, team1),
        "team1_win_pct_at_venue": venue_win_pct(history, team1, venue),
        "team2_win_pct_at_venue": venue_win_pct(history, team2, venue),
        "winner": row["winner"]
    })

feature_df = pd.DataFrame(feature_rows)
feature_df.head(10)

In [4]:
feature_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 256 entries, 0 to 255
Data columns (total 15 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   match_id                    256 non-null    object        
 1   match_date                  256 non-null    datetime64[ns]
 2   team1                       256 non-null    object        
 3   team2                       256 non-null    object        
 4   venue                       256 non-null    object        
 5   season                      256 non-null    int64         
 6   toss_winner                 256 non-null    object        
 7   toss_decision               256 non-null    object        
 8   team1_win_pct_last_5        256 non-null    float64       
 9   team2_win_pct_last_5        256 non-null    float64       
 10  team1_head_to_head_win_pct  256 non-null    float64       
 11  team2_head_to_head_win_pct  256 non-null    float64       

In [5]:
feature_df.tail(10)

    match_id match_date  ... team2_win_pct_at_venue                       winner
246  1473465 2025-04-13  ...               0.000000  Royal Challengers Bengaluru
247  1473466 2025-04-13  ...               0.500000               Mumbai Indians
248  1473467 2025-04-14  ...               0.000000          Chennai Super Kings
249  1473468 2025-04-15  ...               0.500000                 Punjab Kings
250  1473469 2025-04-16  ...               0.000000                         None
251  1473470 2025-04-17  ...               0.550000               Mumbai Indians
252  1473471 2025-04-18  ...               0.000000                 Punjab Kings
253  1473472 2025-04-19  ...               0.578947               Gujarat Titans
254  1473473 2025-04-19  ...               0.454545         Lucknow Super Giants
255  1473474 2025-04-20  ...               0.500000  Royal Challengers Bengaluru

[10 rows x 15 columns]